In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import torch
from velovi import preprocess_data, VELOVI
import matplotlib.pyplot as plt
import seaborn as sns
import pickle as pickle
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
def add_velovi_outputs_to_adata(adata, vae):
    latent_time = vae.get_latent_time(n_samples=25)
    velocities = vae.get_velocity(n_samples=25, velo_statistic="mean")

    t = latent_time
    scaling = 20 / t.max(0)

    adata.layers["velocity"] = velocities / scaling
    adata.layers["latent_time_velovi"] = latent_time

    adata.var["fit_alpha"] = vae.get_rates()["alpha"] / scaling
    adata.var["fit_beta"] = vae.get_rates()["beta"] / scaling
    adata.var["fit_gamma"] = vae.get_rates()["gamma"] / scaling
    adata.var["fit_t_"] = (
        torch.nn.functional.softplus(vae.module.switch_time_unconstr)
        .detach()
        .cpu()
        .numpy()
    ) * scaling
    adata.layers["fit_t"] = latent_time.values * scaling[np.newaxis, :]
    adata.var['fit_scaling'] = 1.0

## Gastrulation erythroid

In [3]:
adata = sc.read_h5ad('../Data/gastrulation_erythroid.h5ad')
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

adata = preprocess_data(adata)
VELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = VELOVI(adata)
vae.train()
add_velovi_outputs_to_adata(adata, vae)

scv.tl.velocity_graph(adata, vkey='velocity', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velocity')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity', color='celltype', title='VeloVI', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Gastrulation_Erythroid_Results/VeloVI_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Gastrulation_Erythroid_Results/VeloVI.h5ad')

Filtered out 47456 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.


/data/conda_env/lyw/scVI/lib/python3.9/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
/data/conda_env/lyw/scVI/lib/python3.9/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps_array.dtype], [])
/data/conda_env/lyw/scVI/lib/python3.9/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more infor

Extracted 3000 highly variable genes.
Logarithmized X.
computing neighbors


/data/conda_env/lyw/scVI/lib/python3.9/site-packages/scvelo/preprocessing/utils.py:705: DeprecationWarning: `log1p` is deprecated since scVelo v0.3.0 and will be removed in a future version. Please use `log1p` from `scanpy.pp` instead.
  log1p(adata)
/tmp/ipykernel_2952085/988312166.py:4: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
/data/conda_env/lyw/scVI/lib/python3.9/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(


    finished (0:00:23) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:02) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocities
    finished (0:00:01) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)


Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA H100 80GB HBM3') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Epoch 279/500:  56%|██▏ | 279/500 [01:55<01:31,  2.41it/s, v_num=1, train_loss_step=-1.12e+3, train_loss_epoch=-1.07e+3]
Monitored metric elbo_validation did not improve in the last 45 records. Best score: -1013.940. Signaling Trainer to stop.
computing velocity graph (using 192/192 cores)
or disable the progress bar using `show_progress_bar=False`.
    finished (0:00:11) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_umap', embedded velocity vectors (adata.obsm)


## Dentate Gyrus neurogenesis

In [4]:
adata = sc.read_h5ad('../Data/DentateGyrus_10X43_1.h5ad')
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

adata = preprocess_data(adata)
VELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = VELOVI(adata)
vae.train()
add_velovi_outputs_to_adata(adata, vae)

scv.tl.velocity_graph(adata, vkey='velocity', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velocity')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity', color='clusters', title='VeloVI', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../GDentate_Gyrus_Results/VeloVI_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Dentate_Gyrus_Results/VeloVI.h5ad')

Filtered out 10340 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.
Extracted 3000 highly variable genes.
Logarithmized X.


/data/conda_env/lyw/scVI/lib/python3.9/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
/data/conda_env/lyw/scVI/lib/python3.9/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps_array.dtype], [])
/data/conda_env/lyw/scVI/lib/python3.9/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more infor

computing neighbors
    finished (0:00:00) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocities


Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
Epoch 310/500:  62%|███  | 310/500 [00:42<00:26,  7.29it/s, v_num=1, train_loss_step=-2.1e+3, train_loss_epoch=-2.16e+3]
Monitored metric elbo_validation did not improve in the last 45 records. Best score: -2036.388. Signaling Trainer to stop.
computing velocity graph (using 192/192 cores)
    finished (0:00:13) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_umap', embedded velocity vectors (adata.obsm)


## Simulated data

In [6]:
adata = sc.read_h5ad('../Data/simulated_T_cell.h5ad')
adata.obsm['X_umap'] = pickle.load(open('../Data/UMAP.pkl','rb'))
scv.pp.moments(adata, n_neighbors=30)

adata = preprocess_data(adata, min_max_scale=True, filter_on_r2=False)
VELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = VELOVI(adata)
vae.train()
add_velovi_outputs_to_adata(adata, vae)

scv.tl.velocity_graph(adata, vkey='velocity', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velocity')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity', color='time', title='VeloVI', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Simulation_Results/VeloVI_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Simulation_Results/VeloVI.h5ad')

/tmp/ipykernel_2952085/3616980057.py:7: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be removed in SciPy 2.0.0.
  adata_umap = pickle.load(open(f'/home/ylz0045/Singlecell/RNA_velocity/our_perspective_TME_simulation/TME_simulation/sim1/PAGA/PAGA.pkl','rb'))
/tmp/ipykernel_2952085/3616980057.py:9: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata, n_neighbors=30)
/data/conda_env/lyw/scVI/lib/python3.9/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(


computing neighbors


Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


    finished (0:00:00) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
Epoch 158/500:  32%|███▊        | 158/500 [00:26<00:57,  5.96it/s, v_num=1, train_loss_step=2.71, train_loss_epoch=-1.6]
Monitored metric elbo_validation did not improve in the last 45 records. Best score: -0.207. Signaling Trainer to stop.
computing velocity graph (using 192/192 cores)
    finished (0:00:02) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_umap', embedded velocity vectors (adata.obsm)
